<a href="https://colab.research.google.com/github/rafaellopesdesa/hnsbi-toolkit/blob/main/examples/dingo_bbh/dingo_bbh_dual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reduced DINGO-inspired BBH dual workflow

This is a complete, opt-in dual hNPE--hNDE workflow on a reduced analytic benchmark. It trains and reloads all five artifacts, evaluates the posterior and likelihood routes on common proposal points, forms their dual consensus, checks normalization and route agreement, and compares with the exact reduced Gaussian likelihood. It does **not** run DINGO or perform production gravitational-wave inference.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# The only execution switch. False performs a fast data/configuration smoke
# check. True generates the quick dataset and runs training plus inference.
RUN_FULL_WORKFLOW = False
PROFILE = "quick" if RUN_FULL_WORKFLOW else "smoke"

IN_COLAB = "google.colab" in sys.modules
ROOT = Path("/content/hnsbi-toolkit") if IN_COLAB else Path.cwd()
if IN_COLAB and not ROOT.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/rafaellopesdesa/hnsbi-toolkit.git", str(ROOT)],
        check=True,
    )
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[bayes,plots]"],
        check=True,
    )
os.chdir(ROOT)
print(f"RUN_FULL_WORKFLOW={RUN_FULL_WORKFLOW}; data profile={PROFILE}")

In [ ]:
subprocess.run(
    [
        sys.executable,
        "examples/dingo_bbh/generate_data.py",
        "--output-dir",
        "examples/dingo_bbh/data",
        "--profile",
        PROFILE,
    ],
    check=True,
)

from hnsbi import Project

project = Project.load("examples/dingo_bbh/dual.yaml")
training_data = project.dual_training_data()
print("rho:", training_data.rho_flow.theta.shape, training_data.rho_flow.observation.shape)
print("nu:", training_data.nu_flow.theta.shape, training_data.nu_flow.observation.shape)

In [ ]:
import importlib.util
import numpy as np

generator_path = ROOT / "examples" / "dingo_bbh" / "generate_data.py"
specification = importlib.util.spec_from_file_location("dingo_bbh_generator", generator_path)
generator = importlib.util.module_from_spec(specification)
specification.loader.exec_module(generator)

truth_theta = training_data.validation.theta[:1]
observed = training_data.validation.observation[:1]
print("truth theta:", dict(zip(generator.THETA_FEATURES, truth_theta[0], strict=True)))
print("exact log likelihood at truth:", generator.exact_log_likelihood(truth_theta, observed)[0])

With `RUN_FULL_WORKFLOW = True`, the next cell makes the real `Project.train_dual()` call and then reloads the checksummed ONNX bundle. The quick profile is suitable for exercising the workflow, not for publication claims.

In [ ]:
artifacts = None
model = None
rho = project.design_distribution("rho")
if RUN_FULL_WORKFLOW:
    from hnsbi.bayes import load_dual_model

    artifacts = project.train_dual()
    print("saved dual manifest:", artifacts.manifest_path)
    model = load_dual_model(artifacts.manifest_path, rho=rho, verify=True)
    print("reloaded artifacts:", tuple(model.artifacts))
else:
    print("Smoke check complete. Set RUN_FULL_WORKFLOW = True and run all cells for training and inference.")

In [ ]:
if RUN_FULL_WORKFLOW:
    from hnsbi.bayes import (
        WeightedSamples,
        geometric_consensus,
        hnde_log_weights,
        hnpe_log_weights,
        route_diagnostic,
    )

    rng = np.random.default_rng(20260731)
    theta_candidates = model.sample_posterior_denominator(4096, observation=observed, rng=rng)
    log_hnpe = hnpe_log_weights(model, theta_candidates, observed)
    log_hnde = hnde_log_weights(model, theta_candidates, observed)
    hnpe = WeightedSamples.from_log_weights(theta_candidates, log_hnpe, metadata={"route": "hnpe"})
    hnde = WeightedSamples.from_log_weights(theta_candidates, log_hnde, metadata={"route": "hnde"})

    consensus_weights = geometric_consensus(log_hnpe, log_hnde)
    consensus_log_weights = np.full(len(consensus_weights), -np.inf)
    positive = consensus_weights > 0
    consensus_log_weights[positive] = np.log(consensus_weights[positive])
    dual = WeightedSamples(
        theta_candidates,
        consensus_weights,
        consensus_log_weights,
        metadata={"route": "dual"},
    )
    route_report = route_diagnostic(model, theta_candidates, observed)
    print("ESS:", {"hNPE": hnpe.ess, "hNDE": hnde.ess, "dual": dual.ess})
    print("route log-weight RMS:", route_report.log_weight_rms)
    print("hNPE tails:", route_report.hnpe)
    print("hNDE tails:", route_report.hnde)

In [ ]:
if RUN_FULL_WORKFLOW:
    from hnsbi.bayes import (
        bridge_diagnostic,
        conditional_normalization_diagnostic,
        estimate_evidence,
        posterior_normalization_diagnostic,
    )

    posterior_norm = posterior_normalization_diagnostic(
        model, observed, n_reference=2048, rng=rng
    )
    conditional_norm = conditional_normalization_diagnostic(
        model,
        training_data.validation.theta[:8],
        n_reference=256,
        rng=rng,
    )
    evidence_theta = rho.sample(8192, rng=rng)
    evidence = estimate_evidence(
        model,
        observed,
        evidence_theta,
        integration_log_prob=rho,
    )
    bridge = bridge_diagnostic(
        model,
        theta_candidates[:2048],
        observed,
        log_design_evidence=evidence.log_evidence,
    )
    print("E_denominator[r_P]:", posterior_norm)
    print("conditional corrected Z mean:", float(np.mean(conditional_norm.corrected_z)))
    print("conditional correction ESS:", conditional_norm.corrected_ess)
    print("learned log evidence:", evidence.log_evidence, "+/- relative", evidence.relative_mc_error)
    print("bridge median |residual| / RMS:", bridge.median_absolute_residual, bridge.rms_residual)

In [ ]:
if RUN_FULL_WORKFLOW:
    proposal_log_prob = model.posterior_denominator_log_prob(theta_candidates, observed)
    exact_log_weights = (
        model.log_rho(theta_candidates)
        + generator.exact_log_likelihood(theta_candidates, observed)
        - proposal_log_prob
    )
    exact = WeightedSamples.from_log_weights(
        theta_candidates, exact_log_weights, metadata={"route": "exact-reduced"}
    )

    benchmark_theta = training_data.validation.theta[:512]
    exact_log_likelihood = generator.exact_log_likelihood(benchmark_theta, observed)
    learned_log_likelihood = model.log_likelihood(observed, benchmark_theta)
    likelihood_residual = learned_log_likelihood - exact_log_likelihood
    print("likelihood residual mean / RMS:", float(np.mean(likelihood_residual)), float(np.sqrt(np.mean(likelihood_residual**2))))

    exact_mean = np.sum(exact.values * exact.weights[:, None], axis=0)
    prior_width = generator.PRIOR_HIGH - generator.PRIOR_LOW
    for name, posterior in {"hNPE": hnpe, "hNDE": hnde, "dual": dual}.items():
        mean = np.sum(posterior.values * posterior.weights[:, None], axis=0)
        scaled_mean_error = (mean - exact_mean) / prior_width
        affinity = float(np.sum(np.sqrt(posterior.weights * exact.weights)))
        print(name, "ESS=", posterior.ess, "exact affinity=", affinity)
        print("  scaled mean error:", dict(zip(generator.THETA_FEATURES, scaled_mean_error, strict=True)))
    print("exact reduced posterior ESS:", exact.ess)

In [ ]:
if RUN_FULL_WORKFLOW:
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
    for index, (axis, parameter) in enumerate(zip(axes.flat, generator.THETA_FEATURES, strict=False)):
        bins = np.linspace(generator.PRIOR_LOW[index], generator.PRIOR_HIGH[index], 35)
        for name, posterior, color in (
            ("exact", exact, "black"),
            ("hNPE", hnpe, "tab:blue"),
            ("hNDE", hnde, "tab:orange"),
            ("dual", dual, "tab:green"),
        ):
            axis.hist(
                posterior.values[:, index],
                bins=bins,
                weights=posterior.weights,
                density=True,
                histtype="step",
                label=name,
                color=color,
            )
        axis.axvline(truth_theta[0, index], color="0.5", linestyle=":")
        axis.set_xlabel(parameter)
    axes.flat[-1].axis("off")
    axes.flat[0].legend()
    figure.suptitle("Reduced BBH posterior: exact benchmark and dual routes")
    plt.show()

## Interpretation

Normalization estimates should be compatible with one within Monte Carlo and surrogate error. High ESS is useful only when the hNPE and hNDE routes agree and both match the independent analytic benchmark. The route RMS, bridge residual, likelihood residual, marginal affinity, and scaled posterior-mean differences are diagnostics—not acceptance thresholds. Increase the data/model profile and repeat with independent seeds before making scientific claims.